The goal of this project is to explore the two libraries with 'relu' activation function and optimizer(solver) on the Climate model simulation crashes dataset (https://archive.ics.uci.edu/dataset/252/climate+model+simulation+crashes) to predict climate model simulation outcomes (fail or succeed) given scaled values of climate model input parameters.

This dataset contains records of simulation crashes encountered during climate model uncertainty quantification (UQ) ensembles. Column 1: Latin hypercube study ID (study 1 to study 3) Column 2: simulation ID (run 1 to run 180) Columns 3-20: values of 18 climate model parameters scaled in the interval [0, 1] Column 21: simulation outcome (0 = failure, 1 = success)

The MLPClassifier is a library from the sklearn.neural_network module in the Scikit-learn framework, which provides tools for machine learning, including simple neural networks. On the other hand, TensorFlow is a deep learning framework designed for creating more complex and scalable neural network models.

We will explore three solvers, SGD (Stochastic Gradient Descent), Adam (Adaptive Moment Estimation), and L-BFGS (Limited-memory Broyden-Fletcher-Goldfarb-Shanno). These were selected to check how accurately they can perform with stable prediction.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sn

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import train_test_split, GridSearchCV,  RandomizedSearchCV
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

from sklearn.model_selection import cross_val_score

from sklearn.pipeline import Pipeline

import tensorflow as tf


In [2]:
# Correctly define file paths
input_file = 'pop_failures.dat'  # Replace with the actual path to your .dat file
output_csv = 'pop_failures.csv'  # Specify where to save the CSV file

# Load the .dat file with space-separated values
data = pd.read_csv(input_file, delim_whitespace=True)

# Save to CSV
data.to_csv(output_csv, index=False)
print(f"Data has been converted and saved to {output_csv}")

C:\Users\Admin\AppData\Local\Temp\ipykernel_28044\4196777611.py:6: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  data = pd.read_csv(input_file, delim_whitespace=True)


PermissionError: [Errno 13] Permission denied: 'pop_failures.csv'

In [ ]:
## import dataset
data = pd.read_csv("pop_failures.csv")

# Print column names
print("data column name:")
print(data.columns)

# Print dimensions of the dataset
print("The dimension of the dataset is:")
print(data.shape)

In [ ]:
data.info()

In [ ]:
data.describe()

In [ ]:
# Group by 'wheatLabel' and count the occurrences of each label to ascertain the state of balancing in the dataset
data_label_count = data.groupby('outcome').size().reset_index(name='count')
print(data_label_count)

data_label_count = data['outcome'].value_counts()
# Plotting
plt.figure(figsize=(8, 5))
data_label_count.plot(kind='bar', color='skyblue')
plt.title("Distribution of climate model success outcome")
plt.xlabel("Outcome")
plt.ylabel("count")
plt.xticks(rotation=0)
plt.savefig("data_label_hist.png")  #save the plot
plt.show()

In [ ]:
# Check if there are any duplicate rows in the dataset
duplicates = data.duplicated()
print("Number of duplicate:", duplicates.sum())

Missing_V = data.isna().sum()
print("Sum of missing value: \n", Missing_V)

In [ ]:
data.hist(figsize=(15,10))
plt.show()



In [ ]:
# Separate target and features
X = data.drop(columns=['outcome','Study', 'Run'])
y = data['outcome']


# Label Encode the target variable (Toilet_type)
le = LabelEncoder()
y_encoded = le.fit_transform(y)

# Split the dataset into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y_encoded, test_size=0.3, random_state=123)

In [ ]:
# Define and evaluate models
def evaluate_models(models, X_train, y_train, X_test, y_test, cv=10):
    results = {}
    for name, model in models.items():
        print(f"\nTraining and evaluating {name}...")
        model.fit(X_train, y_train)  # Train the model
        y_pred = model.predict(X_test)  # Predict on test data
        report = classification_report(y_test, y_pred, output_dict=True, zero_division=1)  # Evaluation report
        results[name] = {
            'accuracy': report['accuracy'],
            'weighted_f1_score': report['weighted avg']['f1-score']
        }
        # Perform cross-validation
        cross_val_accuracies = cross_val_score(estimator=model, X=X_train, y=y_train, cv=cv)
        results[name]['cv_mean_accuracy'] = cross_val_accuracies.mean()
        results[name]['cv_std_accuracy'] = cross_val_accuracies.std()
    return results

# Define the models
models = {
    'model_sgd': MLPClassifier(max_iter=1000, hidden_layer_sizes=(300,), activation='relu', solver='sgd'),
    'model_adam': MLPClassifier(max_iter=1000, hidden_layer_sizes=(300,), activation='relu', solver='adam'),
    'model_lbfgs': MLPClassifier(max_iter=500, hidden_layer_sizes=(300,), activation='relu', solver='lbfgs')
}

# Evaluate the models
results = evaluate_models(models, X_train, y_train, X_test, y_test)

# Display results
for model_name, result in results.items():
    print(f"\nModel: {model_name}")
    print(f"Test Accuracy: {result['accuracy']:.2f}")
    print(f"Weighted F1-score: {result['weighted_f1_score']:.2f}")
    print(f"Cross-validation Accuracy: {result['cv_mean_accuracy']*100:.2f} %")
    print(f"Cross-validation Std Dev: {result['cv_std_accuracy']*100:.2f} %")


## Interpretation of Each Metric
#### Test Accuracy: Measures how the model performs on unseen test data.
L-BFGS (94%) has the highest accuracy, followed by Adam (93%), and SGD (91%).

#### Weighted F1-score uses both precision and recall for imbalanced datasets.
SGD has the lowest F1-score (0.87), meaning it struggles more with class balance compared to Adam (0.93) and L-BFGS (0.94).

#### Cross-validation Accuracy & Std Dev
L-BFGS (94.42%) has the highest average cross-validation accuracy but has the highest variance (3.30%).
Adam (94.17%) has slightly lower accuracy but better stability (2.31% standard deviation).
SGD (91.54%) has the lowest accuracy but is the most stable (1.03% standard deviation).

L-BFGS performs the best in terms of accuracy and F1-score but has the highest variance, meaning its performance is less consistent across different validation folds.
Adam offers a good balance between accuracy and stability, making it a strong candidate for deployment.
SGD has the lowest accuracy and F1 score, but it's the most stable, suggesting it might be more reliable for simpler or smaller datasets.

For this case, we will further adopt Adam and SGD because the sample size of the data is small and we prefer a stable model.

In [ ]:

# Define the parameter grid
param_grid = {
    'mlp__hidden_layer_sizes': [(50, 50), (100,)],
    'mlp__solver': ['sgd', 'adam'],
    'mlp__alpha': [0.0001, 0.001],
    'mlp__learning_rate': ['constant', 'adaptive'],
    'mlp__learning_rate_init': [0.001, 0.01]
}

# Define a pipeline
pipeline = Pipeline([
    #('scaler', StandardScaler()),  # Normalize the features
    ('mlp', MLPClassifier(max_iter=2000))
])

# Perform grid search
grid = GridSearchCV(pipeline, param_grid, scoring='f1_weighted')
grid.fit(X_train, y_train)

# Best parameters
print(grid.best_params_)

best_accuracy = grid.best_score_
best_parameters = grid.best_params_

# Best parameters
print("Best Accuracy: {:.2f} %".format(best_accuracy*100))
print("Best Parameters:", best_parameters)


Adam optimizer outperforms SGD because it adapts learning rates for different parameters and the imbalance class nature of the dataset

A single hidden layer (100 neurons) is better than (50,50) because it captures patterns efficiently without adding unnecessary complexity.

The adaptive learning rate is better than constant as the model adjusts its learning rate when needed, preventing poor convergence.

L2 Regularization (alpha=0.0001)  shows that small regularization helps prevent overfitting but still allows flexibility.

In [ ]:
best_model = MLPClassifier(
    hidden_layer_sizes=(100,),
    solver='adam',
    alpha=0.0001,
    learning_rate='adaptive',
    learning_rate_init=0.001,
    max_iter=2000
)

best_model.fit(X_train, y_train)

In [ ]:
# Define the parameter grid
param_grid = {
    'mlp__hidden_layer_sizes': [(50, 50), (100,)],
    'mlp__solver': ['sgd', 'adam'],
    'mlp__alpha': [0.0001, 0.001],
    'mlp__learning_rate': ['constant', 'adaptive'],
    'mlp__learning_rate_init': [0.001, 0.01]
}

# Define a pipeline
pipeline = Pipeline([
    ('mlp', MLPClassifier(max_iter=2000))
])

# Perform grid search
grid = RandomizedSearchCV(pipeline, param_grid, scoring='accuracy', cv = 10, n_jobs = -1 )
grid.fit(X_train, y_train)
best_accuracy = grid.best_score_
best_parameters = grid.best_params_

# Best parameters
print("Best Accuracy: {:.2f} %".format(best_accuracy*100))
print("Best Parameters:", best_parameters)

## TENSORFLOW

# CASE with BALANCE CLASS DATASET